# EDA On `player_career_successes.csv` Dataset

This dataset has 1 row per rookie in our cohort. It describes exposure (num off plays, def plays, st plays) and quanitifes success with number of pro bowls/first team all pro/second team all pro.


In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load Data
# ==========================================

pcs_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "player_career_successes.csv"), 
    )
)

# ------------------------------------
# Add Player Names For EDA
# ------------------------------------

players_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "players.csv"), 
        # Change "NA" to None
        null_values = {
            "draft_round": "NA", 
            "draft_pick_within_round": "NA", 
            "draft_overall_pick": "NA"
        }
    )
    .select("nfl_id", "display_name")
)

pcs_df = (
    pcs_df
    .join(
        players_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

print("Player Career Successes Data:")
print(f"    - Dimensions: {pcs_df.shape}")
print(f"    - Number of Unique Player IDs: {pcs_df.select("nfl_id").n_unique()}")
pcs_df

In [ ]:
# ========================================
# Missing Data
# ========================================

# No missing data

pcs_df.select(pl.all().is_null().sum())

In [ ]:
# ======================================
# (EXTRA) General Number of Snaps
# ======================================

snap_columns = [
    "career_offensive_snaps",
    "career_defensive_snaps",
    "career_special_teams_snaps",
]

# Add total number of snaps for each player
pcs_df = pcs_df.with_columns(total_num_snaps = pl.sum_horizontal(snap_columns))

# Has every player played at least 1 snap?
print(f"Number of Players That Have Played 0 Snaps:             {pcs_df.filter(pl.col("total_num_snaps") <= 0).height}")
print(f"Number of Players That Have Played At Least 1    Snap:  {pcs_df.filter(pl.col("total_num_snaps") > 0).height}")
print(f"Number of Players That Have Played At Least 20   Snaps: {pcs_df.filter(pl.col("total_num_snaps") > 20).height}")
print(f"Number of Players That Have Played At Least 50   Snaps: {pcs_df.filter(pl.col("total_num_snaps") > 50).height}")
print(f"Number of Players That Have Played At Least 100  Snaps: {pcs_df.filter(pl.col("total_num_snaps") > 100).height}")
print(f"Number of Players That Have Played At Least 1000 Snaps: {pcs_df.filter(pl.col("total_num_snaps") > 1000).height}")

fig = px.violin(
    pcs_df.filter(pl.col("total_num_snaps") > 0), 
    y = "total_num_snaps", 
    points = "all",
    title = "Total Number of Snaps Played Among Players With >= 1 Snap Played"
)

fig.show()

In [ ]:
# ======================================
# (EXTRA) General Number of Snaps CONTINUED
# ======================================

snap_density_df = pcs_df.filter(pl.col("total_num_snaps") > 0).unpivot(
    on=snap_columns,
    variable_name="snap_type",
    value_name="num_snaps",
)

fig = px.violin(
    snap_density_df,
    y="num_snaps",
    x="snap_type",
    color="snap_type",
    points="all",
    title="Career Snap Counts by Snap Type",
)

fig.update_layout(showlegend=False)

fig.show()

In [ ]:
# =======================================
# career_offensive_snaps
# =======================================

pcs_df_off = pcs_df.filter(pl.col("career_offensive_snaps") >= 1)
print(f"  - Min/Max of career_offensive_snaps: [{min(pcs_df_off["career_offensive_snaps"])}, {max(pcs_df_off["career_offensive_snaps"])}]")

In [ ]:
# =======================================
# career_defensive_snaps
# =======================================

pcs_df_def = pcs_df.filter(pl.col("career_defensive_snaps") >= 1)
print(f"  - Min/Max of career_defensive_snaps: [{min(pcs_df_def["career_defensive_snaps"])}, {max(pcs_df_def["career_defensive_snaps"])}]")

In [ ]:
# =======================================
# career_special_teams_snaps
# =======================================

pcs_df_st = pcs_df.filter(pl.col("career_special_teams_snaps") >= 1)
print(f"  - Min/Max of career_special_teams_snaps: [{min(pcs_df_st["career_special_teams_snaps"])}, {max(pcs_df_st["career_special_teams_snaps"])}]")

In [ ]:
# ========================================
# career_games_active
# ========================================

print(f"Number of Players That Have Been Active In At Least 1 Game: {pcs_df.filter(pl.col("career_games_active") > 0).height}")
print(f"Number of Players That Have Not Been Active For Any Games:  {pcs_df.filter(pl.col("career_games_active") <= 0).height}")
print("")

print("Note: A lot of players have either 0 games active, 17 (1 reg season), 34 (2 reg seasons), or 51 (3 reg seasons)")
print(pcs_df.group_by("career_games_active").agg(count = pl.len()).sort(by = "count", descending=True).head(4))
fig = px.violin(
    pcs_df,
    y="career_games_active",
    points="all",
)

fig.show()

In [ ]:
# ========================================
# career_games_started
# ========================================

print(f"Number of Players That Have Started At Least 1 Game: {pcs_df.filter(pl.col("career_games_started") > 0).height}")
print(f"Number of Players That Have Not Started Any Games:   {pcs_df.filter(pl.col("career_games_started") <= 0).height}")

In [ ]:
# ========================================
# ap_all_pro_1st_team
# ========================================

# Players Who Have Gotten All-Pro 1st Team
print(f"Players Who Have Gotten All-Pro 1st Team: {pcs_df.filter(pl.col("ap_all_pro_1st_team") > 0).select("display_name").unique().to_series().to_list()}")
pcs_df.group_by("ap_all_pro_1st_team").agg(count = pl.len())

In [ ]:
# ========================================
# ap_all_pro_2nd_team
# ========================================

print(f"Players Who Have Gotten All-Pro 1st Team: {pcs_df.filter(pl.col("ap_all_pro_2nd_team") > 0).select("display_name").unique().to_series().to_list()}")
pcs_df.group_by("ap_all_pro_2nd_team").agg(count = pl.len())

In [ ]:
# ========================================
# pro_bowl_original_ballot
# ========================================

print(f"Players Who Made Pro-Bowl ONCE:  {pcs_df.filter(pl.col("pro_bowl_original_ballot") == 1).select("display_name").unique().to_series().to_list()}")
print(f"Players Who Made Pro-Bowl TWICE: {pcs_df.filter(pl.col("pro_bowl_original_ballot") == 2).select("display_name").unique().to_series().to_list()}")
pcs_df.group_by("pro_bowl_original_ballot").agg(count = pl.len())